##### Human-Adipose-Depot-DIA (depot difference analysis)
##### Yue (Winnie) Wen, Alex Zelter, Mike Riffle, Nina Isoherranen
##### Department of Pharmaceutics, Department of Genome Science, University of Washington-Seattle
##### 03/07/2025

In [2]:
import pandas as pd
from scipy import stats
from scipy.stats import shapiro
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sbn
import statsmodels.api as sm

##### I. Import Data File & Data Cleaning

In [28]:
# Read level 3b data file
rawdata = pd.read_csv(r"level-3b-data.tsv",sep='\t')
# filter only human protein
rawdata = rawdata[rawdata.protein.str.find('HUMAN') != -1]
rawdata.head(3)

,protein,X015_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3528.SQ.1.lean_A1,X016_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3525.SQ.1.obese_A2,X017_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3521.OM.2.lean_A3,X018_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3523.SQ.4.obese_A4,X019_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3518.SQ.2.obese_A5,X020_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3530.SQ.2.lean_A6,X021_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3505.SQ.2.obese_A7,X022_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3506.OM.2.obese_A8,X023_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3507.OM.1.obese_A9,...,X106_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3504.SQ.4.lean_E2_low_12ug,X107_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3520.SQ.4.obese_E3,X108_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3531.SQ.4.obese_E4,X109_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3508.SQ.1.obese_E5,X110_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3518.OM.1.obese_E6,X111_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3507.SQ.4.obese_E7,X112_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3509.OM.1.obese_E8,X113_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3513.SQ.4.lean_E9,X114_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3524.SQ.1.obese_E10,X115_Dora_Neo_2024_0311_adipose_Exp1_15cm_neopepsep_60m_51.3510.SQ.1.obese_E11
0,sp|A0A075B6H7|KV37_HUMAN,26.313764,24.119750,25.113549,24.261524,23.985577,24.046824,24.048730,24.799880,25.577012,...,25.283767,25.152245,24.135758,23.259427,23.980391,23.981357,24.576698,23.781046,23.661355,25.628251
1,sp|A0A075B6H9|LV469_HUMAN,22.720586,24.299526,23.590884,25.011210,24.872071,24.212893,24.331776,23.492445,24.965317,...,25.255538,24.456239,25.326547,23.518094,24.360569,23.633028,23.909291,24.133126,24.596189,24.881849
2,sp|A0A075B6I0|LV861_HUMAN,26.377009,24.349982,25.071946,26.696202,27.211654,26.388782,26.162725,25.514984,26.563370,...,26.007549,25.718241,26.179879,25.064865,26.133463,25.342857,25.912140,25.851064,25.463426,27.281809


In [29]:
def getCleanColumnName(input):
    split_lst = input.split('.')
    return split_lst[1] + "_" + split_lst[2] + "_" + split_lst[4].split('_')[0]

In [30]:
# Data cleaning
for i in rawdata.columns[1:]:
    rawdata=rawdata.rename(columns={i:getCleanColumnName(i)})

rawdata = pd.melt(rawdata,id_vars="protein", value_vars=rawdata.columns[1:])

rawdata["raw_value"] = rawdata["value"].copy() #raw_value is the column with log2(+1) directLFQ normalized value
rawdata["value"] = 2**(rawdata["value"]) #didn't -1 here to bypass 0 issue with further statistical analysis 

rawdata['tissue_type'] = rawdata['variable'].str.split('_').str[1]
rawdata['bmi'] = rawdata['variable'].str.split('_').str[2]
rawdata = rawdata.rename(columns={'variable':'id'})
rawdata['id'] = rawdata['id'].str.split('_').str[0]

In [31]:
rawdata.to_csv('figure-6-7-8-9-data.csv', index=False, mode="w")

##### II. OM-SQ Depot Differences - Statistical Testing

In [40]:
# The paired Wilcoxon-signed rank test only include participants that have both OM and SQ adipose tissue available in the study.
Should_include_participants = [3501,3502,3503,3504,3505,3506,3507,
                               3508,3509,3510,3512,3514,3515,
                               3516,3517,3518,3519,3520,3521,3522,
                               3523,3524,3525,3527,3528,3529,
                               3531,3532]
cleandata = rawdata.copy()
cleandata['id'] = cleandata['id'].astype(int)
cleandata = cleandata[cleandata.id.isin(Should_include_participants)]

In [42]:
tested_protein_list = []
test_statistics_list = []
OM_mean_PA = []
SC_mean_PA = []
p_value_list = []
fold_change_list = []
for i in cleandata['protein'].unique():
    temp = pd.pivot_table(cleandata[cleandata.protein==i], values='value', index='id', columns='tissue_type')
    stat1,p1 = stats.wilcoxon(temp.OM, temp.SQ, nan_policy='omit', zero_method='pratt')
    OM_mean_PA.append(np.mean(temp.OM))
    SC_mean_PA.append(np.mean(temp.SQ))
    test_statistics_list.append(stat1)
    p_value_list.append(p1)
    tested_protein_list.append(i)
    fold_change_list.append(np.mean(np.log2(temp.SQ/temp.OM)))
OM_SC_paired_tests = pd.DataFrame({"Protein":tested_protein_list, "Mean OM Protein Peak Area":OM_mean_PA, "Mean SC Protein Peak Area": SC_mean_PA, "log2(Fold Change)":fold_change_list, "P Value":p_value_list})
OM_SC_paired_tests.head(3)

c:\Users\Winnie\anaconda3\Lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
c:\Users\Winnie\anaconda3\Lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
c:\Users\Winnie\anaconda3\Lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "
c:\Users\Winnie\anaconda3\Lib\site-packages\scipy\stats\_morestats.py:4088: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does 

,Protein,Mean OM Protein Peak Area,Mean SC Protein Peak Area,log2(Fold Change),P Value
0,sp|A0A075B6H7|KV37_HUMAN,2.587528e+07,3.166254e+07,0.138369,0.327210
1,sp|A0A075B6H9|LV469_HUMAN,1.977795e+07,2.312047e+07,0.198810,0.235917
2,sp|A0A075B6I0|LV861_HUMAN,6.856913e+07,7.491892e+07,0.174482,0.451477


In [ ]:
# Multiple hypothesis correction
OM_SC_paired_tests["Adjusted P Value"] = stats.false_discovery_control(OM_SC_paired_tests["P Value"])
OM_SC_paired_tests.head(3)

,Protein,Mean OM Protein Peak Area,Mean SC Protein Peak Area,log2(Fold Change),P Value,Adjusted P Value
0,sp|A0A075B6H7|KV37_HUMAN,2.587528e+07,3.166254e+07,0.138369,0.327210,0.465406
1,sp|A0A075B6H9|LV469_HUMAN,1.977795e+07,2.312047e+07,0.198810,0.235917,0.363924
2,sp|A0A075B6I0|LV861_HUMAN,6.856913e+07,7.491892e+07,0.174482,0.451477,0.588121


In [44]:
OM_SC_paired_tests = OM_SC_paired_tests.sort_values(by="Adjusted P Value", ascending=True)

In [49]:
OM_SC_paired_tests.to_csv('supplemental_table1_protein_level_overview.csv', index=False, mode="w")

In [50]:
len(OM_SC_paired_tests[OM_SC_paired_tests["Adjusted P Value"]<0.05].Protein.unique())

1660

In [51]:
len(OM_SC_paired_tests[(OM_SC_paired_tests["Adjusted P Value"]<0.05)&(OM_SC_paired_tests["log2(Fold Change)"]>1)].Protein.unique())

93

In [52]:
len(OM_SC_paired_tests[(OM_SC_paired_tests["Adjusted P Value"]<0.05)&(OM_SC_paired_tests["log2(Fold Change)"]<-1)].Protein.unique())

200